# Feature Layer Validation

This notebook validates the sanity and quality of engineered feature datasets produced by FeatureDealing.ipynb.

## Validation Checks:
1. Load and describe feature tables
2. Check for missing values and anomalies
3. Analyze feature distributions
4. Verify train/test split integrity
5. Check feature correlations
6. Validate target variable (resale_price)
7. Cross-validate with raw data

## Section 1: Setup and Import

In [1]:
import json
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
from pathlib import Path
from datetime import datetime

SEED = 42
np.random.seed(SEED)
pd.set_option('display.max_columns', None)
pd.set_option('display.max_colwidth', 150)

ROOT = Path('/Users/lorenzolou/Library/Mobile Documents/com~apple~CloudDocs/NUS/PropertyLens')
FEATURE_DIR = ROOT / '02_feature_layer' / 'training'
OUTPUT_DIR = FEATURE_DIR / 'outputs'
RAW_DIR = ROOT / '01_data_layer' / 'raw'

print('Feature directory:', FEATURE_DIR)
print('Output directory:', OUTPUT_DIR)
print(f'Available output files: {len(list(OUTPUT_DIR.glob("*.csv")))}')

Feature directory: /Users/lorenzolou/Library/Mobile Documents/com~apple~CloudDocs/NUS/PropertyLens/02_feature_layer/training
Output directory: /Users/lorenzolou/Library/Mobile Documents/com~apple~CloudDocs/NUS/PropertyLens/02_feature_layer/training/outputs
Available output files: 6


## Section 2: Load Latest Feature Files

In [2]:
def latest_file_by_pattern(folder: Path, pattern: str) -> Path | None:
    files = sorted(folder.glob(pattern))
    return files[-1] if files else None

# Load latest feature files
feature_table_fp = latest_file_by_pattern(OUTPUT_DIR, 'hdb_feature_table_*.csv')
train_fp = latest_file_by_pattern(OUTPUT_DIR, 'hdb_feature_train_*.csv')
test_fp = latest_file_by_pattern(OUTPUT_DIR, 'hdb_feature_test_*.csv')
metadata_fp = latest_file_by_pattern(OUTPUT_DIR, 'feature_metadata_*.json')

if not all([feature_table_fp, train_fp, test_fp, metadata_fp]):
    raise FileNotFoundError('Missing one or more feature output files')

# Load data
feature_table = pd.read_csv(feature_table_fp)
train_df = pd.read_csv(train_fp)
test_df = pd.read_csv(test_fp)

with open(metadata_fp, 'r') as f:
    metadata = json.load(f)

print('Feature Table loaded:', feature_table_fp.name)
print('Train set loaded:', train_fp.name)
print('Test set loaded:', test_fp.name)
print('\nDataset shapes:')
print(f'  Feature table: {feature_table.shape}')
print(f'  Train: {train_df.shape}')
print(f'  Test: {test_df.shape}')
print(f'\nMetadata: {json.dumps(metadata, indent=2)}')

Feature Table loaded: hdb_feature_table_20260406.csv
Train set loaded: hdb_feature_train_20260406.csv
Test set loaded: hdb_feature_test_20260406.csv

Dataset shapes:
  Feature table: (260699, 79)
  Train: (178589, 79)
  Test: (82110, 79)

Metadata: {
  "export_date": "20260406",
  "total_rows": 260699,
  "train_rows": 178589,
  "test_rows": 82110,
  "columns": 79,
  "split_year": 2023,
  "duplicates_removed": 2305
}


## Section 3: Data Quality Overview

In [3]:
print("="*70)
print("DATA QUALITY OVERVIEW")
print("="*70)

# Check for duplicates
total_rows = len(feature_table)
unique_rows = len(feature_table.drop_duplicates())
dup_count = total_rows - unique_rows

print(f"\n1. DUPLICATE ROWS")
print(f"   Total rows: {total_rows:,}")
print(f"   Unique rows: {unique_rows:,}")
print(f"   Duplicate rows: {dup_count:,}")
if dup_count > 0:
    print(f"   ⚠️  WARNING: {dup_count} duplicate rows found")
else:
    print(f"   ✓ No duplicates found")

# Check for missing values
print(f"\n2. MISSING VALUES")
missing = feature_table.isnull().sum()
missing_pct = 100 * missing / len(feature_table)
missing_df = pd.DataFrame({
    'Column': missing.index,
    'Missing Count': missing.values,
    'Missing %': missing_pct.values
}).sort_values('Missing %', ascending=False)

if missing.sum() > 0:
    print(f"   Total missing values: {missing.sum()}")
    print(f"   Columns with missing data:")
    print(missing_df[missing_df['Missing Count'] > 0].to_string(index=False))
else:
    print(f"   ✓ No missing values found")

# Check data types
print(f"\n3. DATA TYPES & RANGES")
for col in feature_table.columns:
    if col == 'resale_price':  # Target variable
        min_val = feature_table[col].min()
        max_val = feature_table[col].max()
        mean_val = feature_table[col].mean()
        print(f"   {col}: [{min_val:,.0f}, {max_val:,.0f}], mean={mean_val:,.0f}")

DATA QUALITY OVERVIEW

1. DUPLICATE ROWS
   Total rows: 260,699
   Unique rows: 260,699
   Duplicate rows: 0
   ✓ No duplicates found

2. MISSING VALUES
   ✓ No missing values found

3. DATA TYPES & RANGES
   resale_price: [140,000, 1,700,000], mean=514,748


## Section 4: Feature Distribution Analysis

In [4]:
print("="*70)
print("FEATURE DISTRIBUTION ANALYSIS")
print("="*70)

# Get numeric columns (excluding target and categorical dummies)
numeric_cols = feature_table.select_dtypes(include=[np.number]).columns
core_factors = [
    'level_mid', 'lease_remaining_years', 'floor_area_sqm', 'room_count',
    'dist_to_mrt_m', 'orientation_score', 'dist_to_highway_m', 'dist_to_foodcourt_m',
    'mall_count_3km', 'mall_weighted_access_3km',
    'dist_to_nearest_school_m', 'school_count_1km', 'primary_school_quality_1km_weighted'
]

# Statistics for core factors
print(f"\nCore Factor Statistics (Engineered Features):")
print("-" * 70)

stats = []
for col in core_factors:
    if col in feature_table.columns:
        data = feature_table[col]
        stats.append({
            'Factor': col,
            'Count': data.notna().sum(),
            'Mean': data.mean(),
            'Std': data.std(),
            'Min': data.min(),
            'Max': data.max(),
            'Nulls': data.isna().sum()
        })

stats_df = pd.DataFrame(stats)
print(stats_df.to_string(index=False))

# Check for outliers in price
print(f"\n\nTarget Variable (resale_price) Analysis:")
print("-" * 70)
price = feature_table['resale_price']
q1 = price.quantile(0.25)
q3 = price.quantile(0.75)
iqr = q3 - q1
lower_bound = q1 - 1.5 * iqr
upper_bound = q3 + 1.5 * iqr
outliers = ((price < lower_bound) | (price > upper_bound)).sum()

print(f"  Q1: ${q1:,.0f}")
print(f"  Q3: ${q3:,.0f}")
print(f"  IQR: ${iqr:,.0f}")
print(f"  Outliers (1.5*IQR): {outliers:,} rows ({100*outliers/len(price):.2f}%)")
print(f"  Price range: ${price.min():,.0f} - ${price.max():,.0f}")
print(f"  Mean: ${price.mean():,.0f}")
print(f"  Median: ${price.median():,.0f}")

FEATURE DISTRIBUTION ANALYSIS

Core Factor Statistics (Engineered Features):
----------------------------------------------------------------------
                             Factor  Count        Mean         Std       Min          Max  Nulls
                          level_mid 260699    8.689393    5.875013  2.000000    50.000000      0
              lease_remaining_years 260699   74.178643   13.762616 39.000000    98.000000      0
                     floor_area_sqm 260699   96.879229   24.047396 31.000000   366.700000      0
                         room_count 260699    4.037917    0.797881  1.000000     6.000000      0
                      dist_to_mrt_m 260699 2085.368934 2025.680578 22.786157  8275.457242      0
                  orientation_score 260699    0.000533    1.000002 -1.000000     1.000000      0
                  dist_to_highway_m 260699 3963.916183 2517.136645 46.896445 10655.900452      0
                dist_to_foodcourt_m 260699  945.380667  574.856840 34.864206

## Section 5: Train/Test Split Validation

In [5]:
print("="*70)
print("TRAIN/TEST SPLIT VALIDATION")
print("="*70)

train_size = len(train_df)
test_size = len(test_df)
total_size = train_size + test_size
train_pct = 100 * train_size / total_size
test_pct = 100 * test_size / total_size

print(f"\n1. SPLIT PROPORTIONS")
print(f"   Train: {train_size:,} rows ({train_pct:.1f}%)")
print(f"   Test: {test_size:,} rows ({test_pct:.1f}%)")
print(f"   Total: {total_size:,} rows")

if abs(train_pct - 80) < 1 and abs(test_pct - 20) < 1:
    print(f"   ✓ Split proportions correct (80/20)")
else:
    print(f"   ⚠️  WARNING: Split proportions unexpected")

# Check for overlap
print(f"\n2. SPLIT INTEGRITY")
train_cols = set(train_df.columns)
test_cols = set(test_df.columns)
feature_cols = set(feature_table.columns)

if train_cols == test_cols == feature_cols:
    print(f"   ✓ All sets have identical columns")
else:
    print(f"   ⚠️  WARNING: Column mismatch detected")
    print(f"   Feature table columns: {len(feature_cols)}")
    print(f"   Train columns: {len(train_cols)}")
    print(f"   Test columns: {len(test_cols)}")

# Check for data leakage (row overlap)
print(f"\n3. CHECK FOR DATA LEAKAGE")
# Compare key identifiers to check for overlap
if 'address_key' in train_df.columns and 'address_key' in test_df.columns:
    # Note: same address can appear in multiple transactions, so some overlap is expected
    print(f"   Train unique addresses: {train_df['address_key'].nunique():,}")
    print(f"   Test unique addresses: {test_df['address_key'].nunique():,}")
    print(f"   Overlap: {len(set(train_df['address_key']) & set(test_df['address_key'])):,} addresses")
    print(f"   ✓ Address overlap expected (same address, different times)")

# Compare target distribution
print(f"\n4. TARGET DISTRIBUTION ACROSS SPLITS")
train_price = train_df['resale_price']
test_price = test_df['resale_price']

print(f"   Train - Mean: ${train_price.mean():,.0f}, Median: ${train_price.median():,.0f}")
print(f"   Test  - Mean: ${test_price.mean():,.0f}, Median: ${test_price.median():,.0f}")
print(f"   Difference: {abs(train_price.mean() - test_price.mean()) / train_price.mean() * 100:.2f}%")

if abs(train_price.mean() - test_price.mean()) / train_price.mean() < 0.05:
    print(f"   ✓ Target distribution similar between train/test")
else:
    print(f"   ⚠️  WARNING: Target distribution differs between splits")

TRAIN/TEST SPLIT VALIDATION

1. SPLIT PROPORTIONS
   Train: 178,589 rows (68.5%)
   Test: 82,110 rows (31.5%)
   Total: 260,699 rows
   ⚠️  WARNING: Split proportions unexpected

2. SPLIT INTEGRITY
   ✓ All sets have identical columns

3. CHECK FOR DATA LEAKAGE
   Train unique addresses: 9,420
   Test unique addresses: 9,421
   Overlap: 9,131 addresses
   ✓ Address overlap expected (same address, different times)

4. TARGET DISTRIBUTION ACROSS SPLITS
   Train - Mean: $468,788, Median: $435,000
   Test  - Mean: $614,711, Median: $590,000
   Difference: 31.13%
   ⚠️  WARNING: Target distribution differs between splits


## Section 6: Feature Correlation Analysis

In [6]:
print("="*70)
print("FEATURE CORRELATION WITH TARGET")
print("="*70)

# Select numeric features (exclude transaction_year and dummy variables)
numeric_features = feature_table.select_dtypes(include=[np.number]).columns.tolist()
# Remove year and dummy variables for cleaner analysis
core_features = [c for c in numeric_features if 'year' not in c.lower() and 'town_' not in c and 'flat_type_' not in c and 'flat_model_' not in c and c != 'resale_price']

# Compute correlations with only the needed columns
data_subset = feature_table[core_features + ['resale_price']].copy()
corr_matrix = data_subset.corr()

# Extract resale_price correlations as a Series
corr_series = corr_matrix['resale_price'].drop('resale_price').sort_values(ascending=False)

print(f"\nCorrelation with resale_price (sorted):")
for feature, corr in corr_series.items():
    strength = "strong" if abs(corr) > 0.5 else "moderate" if abs(corr) > 0.3 else "weak"
    direction = "positive" if corr > 0 else "negative"
    print(f"  {feature:45s}: {corr:7.4f} ({strength} {direction})")

# Highlight top 5 positive and negative
print(f"\n\nTop 5 Positive Correlations:")
for i, (feature, corr) in enumerate(corr_series.head(5).items(), 1):
    print(f"  {i}. {feature}: {corr:.4f}")

print(f"\nTop 5 Negative Correlations:")
for i, (feature, corr) in enumerate(corr_series.tail(5).items(), 1):
    print(f"  {i}. {feature}: {corr:.4f}")

FEATURE CORRELATION WITH TARGET

Correlation with resale_price (sorted):
  floor_area_sqm                               :  0.5665 (strong positive)
  room_count                                   :  0.5654 (strong positive)
  level_mid                                    :  0.3451 (moderate positive)
  mall_count_3km                               :  0.2154 (weak positive)
  mall_weighted_access_3km                     :  0.1902 (weak positive)
  primary_school_top_quality_1km               :  0.0348 (weak positive)
  dist_to_nearest_school_m                     :  0.0310 (weak positive)
  primary_school_quality_1km_weighted          :  0.0287 (weak positive)
  dist_to_foodcourt_m                          : -0.0206 (weak negative)
  school_count_1km                             : -0.0593 (weak negative)
  orientation_score                            : -0.0599 (weak negative)
  dist_to_nearest_mall_m                       : -0.0742 (weak negative)
  primary_school_count_1km                 

## Section 7: Categorical Feature Analysis

In [7]:
print("="*70)
print("CATEGORICAL FEATURE ANALYSIS")
print("="*70)

# Analyze categorical features
categorical_cols = feature_table.select_dtypes(include=['object']).columns

print(f"\nCategorical Features Found: {len(categorical_cols)}")
for col in categorical_cols:
    unique_count = feature_table[col].nunique()
    top_values = feature_table[col].value_counts().head(3)
    print(f"\n  {col}:")
    print(f"    Unique values: {unique_count}")
    print(f"    Top values:")
    for val, count in top_values.items():
        print(f"      - {val}: {count:,}")

# Analyze one-hot encoded columns
encoded_cols = [c for c in feature_table.columns if '_' in c and c not in ['transaction_year']]
print(f"\n\nOne-Hot Encoded Features: {len(encoded_cols)} columns")

town_cols = [c for c in encoded_cols if c.startswith('town_')]
flattype_cols = [c for c in encoded_cols if c.startswith('flat_type_')]
flatmodel_cols = [c for c in encoded_cols if c.startswith('flat_model_')]

print(f"  Towns: {len(town_cols)}")
print(f"  Flat types: {len(flattype_cols)}")
print(f"  Flat models: {len(flatmodel_cols)}")

CATEGORICAL FEATURE ANALYSIS

Categorical Features Found: 1

  address_key:
    Unique values: 9710
    Top values:
      - 308A PUNGGOL WALK: 167
      - 308C PUNGGOL WALK: 151
      - 187 BOON LAY AVE: 142


One-Hot Encoded Features: 78 columns
  Towns: 26
  Flat types: 7
  Flat models: 21


/var/folders/6p/nwg1ljcj77bfnrwnqw_yprf80000gn/T/ipykernel_60655/1624814427.py:6: Pandas4Warning: For backward compatibility, 'str' dtypes are included by select_dtypes when 'object' dtype is specified. This behavior is deprecated and will be removed in a future version. Explicitly pass 'str' to `include` to select them, or to `exclude` to remove them and silence this warning.
See https://pandas.pydata.org/docs/user_guide/migration-3-strings.html#string-migration-select-dtypes for details on how to write code that works with pandas 2 and 3.
  categorical_cols = feature_table.select_dtypes(include=['object']).columns


## Section 8: Cross-Validation with Raw Data

In [8]:
print("="*70)
print("CROSS-VALIDATION WITH RAW DATA")
print("="*70)

# Load raw HDB data to compare
hdb_dir = RAW_DIR / 'ResaleFlatPrices'
hdb_files = sorted(hdb_dir.glob('*.csv'))
hdb_raw_parts = []

for fp in hdb_files:
    if 'backup' not in fp.name:  # Skip backup files
        df = pd.read_csv(fp)
        df['month_dt'] = pd.to_datetime(df['month'], errors='coerce')
        df = df[df['month_dt'].dt.year >= 2015].copy()
        hdb_raw_parts.append(df)

if hdb_raw_parts:
    hdb_raw = pd.concat(hdb_raw_parts, ignore_index=True)
    
    print(f"\nRaw HDB Data (2015+):")
    print(f"  Total rows: {len(hdb_raw):,}")
    print(f"  Unique addresses (block+street): {(hdb_raw['block'].astype(str) + ' ' + hdb_raw['street_name'].astype(str)).nunique():,}")
    
    print(f"\nFeature Table vs Raw Data:")
    print(f"  Feature table rows: {len(feature_table):,}")
    print(f"  Raw data rows: {len(hdb_raw):,}")
    
    # Compare price ranges
    print(f"\nPrice Validation:")
    raw_price = pd.to_numeric(hdb_raw['resale_price'], errors='coerce')
    feature_price = feature_table['resale_price']
    
    print(f"  Raw data price range: ${raw_price.min():,.0f} - ${raw_price.max():,.0f}")
    print(f"  Feature table price range: ${feature_price.min():,.0f} - ${feature_price.max():,.0f}")
    print(f"  Overlap: {(raw_price.min() <= feature_price.min()) and (raw_price.max() >= feature_price.max())}")
    
    if abs(raw_price.mean() - feature_price.mean()) / raw_price.mean() < 0.01:
        print(f"  ✓ Price distributions match")
    else:
        print(f"  ⚠️  Price distributions differ: raw mean=${raw_price.mean():,.0f}, feature mean=${feature_price.mean():,.0f}")
else:
    print("⚠️  Raw HDB data not found")

CROSS-VALIDATION WITH RAW DATA

Raw HDB Data (2015+):
  Total rows: 263,004
  Unique addresses (block+street): 9,710

Feature Table vs Raw Data:
  Feature table rows: 260,699
  Raw data rows: 263,004

Price Validation:
  Raw data price range: $140,000 - $1,700,000
  Feature table price range: $140,000 - $1,700,000
  Overlap: True
  ✓ Price distributions match


## Section 9: Comprehensive Summary Report

In [9]:
print("\n" + "="*70)
print("FEATURE VALIDATION SUMMARY REPORT")
print("="*70)

checks = []

# Check 1: Size
size_ok = len(feature_table) > 1000000
checks.append(('Large dataset size', size_ok, f"{len(feature_table):,} rows"))

# Check 2: No duplicates
dup_ok = len(feature_table) == len(feature_table.drop_duplicates())
checks.append(('No duplicate rows', dup_ok, f"{len(feature_table) - len(feature_table.drop_duplicates())} duplicates"))

# Check 3: No missing values in core columns
core_missing = feature_table[core_features].isnull().sum().sum()
missing_ok = core_missing == 0
checks.append(('No missing values (core features)', missing_ok, f"{core_missing} missing"))

# Check 4: Valid price range
price_ok = (feature_table['resale_price'].min() > 100000) and (feature_table['resale_price'].max() < 2000000)
checks.append(('Valid price range', price_ok, f"${feature_table['resale_price'].min():,.0f}-${feature_table['resale_price'].max():,.0f}"))

# Check 5: Train/test split
split_ok = (len(train_df) > 1500000) and (len(test_df) > 350000)
checks.append(('Train/test sizes', split_ok, f"{len(train_df):,}/{len(test_df):,}"))

# Check 6: Feature count
feature_ok = len(feature_table.columns) > 20
checks.append(('Sufficient features', feature_ok, f"{len(feature_table.columns)} features"))

# Check 7: Categorical encoding
encoded_ok = len([c for c in feature_table.columns if '_' in c and any(x in c for x in ['town', 'flat_type', 'flat_model'])]) > 10
checks.append(('Categorical encoding', encoded_ok, f"{len([c for c in feature_table.columns if '_' in c])} encoded features"))

print("\nValidation Checks:")
for check_name, status, details in checks:
    status_symbol = "✅" if status else "❌"
    print(f"  {status_symbol} {check_name:30s}: {details}")

all_pass = all(status for _, status, _ in checks)
print(f"\n{'='*70}")
if all_pass:
    print("✅ ALL VALIDATION CHECKS PASSED")
else:
    print("⚠️  SOME VALIDATION CHECKS FAILED - REVIEW ABOVE")

print(f"{'='*70}")
print(f"\nValidation timestamp: {datetime.now().isoformat()}")
print(f"Feature file date: {metadata.get('run_date', 'Unknown')}")
print(f"Total features: {len(feature_table.columns)}")
print(f"Total rows: {len(feature_table):,}")


FEATURE VALIDATION SUMMARY REPORT

Validation Checks:
  ❌ Large dataset size            : 260,699 rows
  ✅ No duplicate rows             : 0 duplicates
  ✅ No missing values (core features): 0 missing
  ✅ Valid price range             : $140,000-$1,700,000
  ❌ Train/test sizes              : 178,589/82,110
  ✅ Sufficient features           : 79 features
  ✅ Categorical encoding          : 79 encoded features

⚠️  SOME VALIDATION CHECKS FAILED - REVIEW ABOVE

Validation timestamp: 2026-04-06T10:39:57.613030
Feature file date: Unknown
Total features: 79
Total rows: 260,699


In [10]:
print("\n" + "="*70)
print("DUPLICATE ROOT CAUSE ANALYSIS")
print("="*70)

print(f"\n1. KEY METRICS")
print(f"   Total rows: {len(feature_table):,}")
print(f"   Unique rows (all columns): {len(feature_table.drop_duplicates()):,}")
print(f"   Unique addresses: {feature_table['address_key'].nunique():,}")
print(f"   Avg rows per address: {len(feature_table) / feature_table['address_key'].nunique():.1f}")

print(f"\n2. HYPOTHESIS: Multiple flat models at same address")
# Check if duplicates are same address but different flat configs
addr_duplicates = feature_table.groupby('address_key').size()
print(f"   Addresses with multiple rows: {(addr_duplicates > 1).sum():,}")
print(f"   Addresses with exactly 1 row: {(addr_duplicates == 1).sum():,}")

# Get one address with multiple rows
multi_row_addr = addr_duplicates[addr_duplicates > 1].index[0]
sample_addr_data = feature_table[feature_table['address_key'] == multi_row_addr]
print(f"\n3. SAMPLE: Address '{multi_row_addr}' ({len(sample_addr_data)} rows)")
print(f"   Unique transactions years: {sample_addr_data['transaction_year'].nunique()}")
print(f"   Unique (year, price) combinations: {len(sample_addr_data.groupby(['transaction_year', 'resale_price']))}")
print(f"   All exact duplicates: {len(sample_addr_data) == len(sample_addr_data.drop_duplicates())}")

# Check if the duplication is from flat_model/flat_type diversity
if 'flat_model' in sample_addr_data.columns:
    print(f"   Unique flat_model values: {sample_addr_data['flat_model'].nunique()}")
if 'flat_type' in sample_addr_data.columns:
    print(f"   Unique flat_type values: {sample_addr_data['flat_type'].nunique()}")

# Show one duplicate set
print(f"\n4. ACTUAL DUPLICATE CHECK")
dup_sample = feature_table.drop_duplicates(keep=False).head(1)
print(f"   Showing 1 row that has duplicates:")
print(f"   Columns: {dup_sample.columns.tolist()[:10]}...")

# Check a specific duplicate pattern
from_addr = dup_sample['address_key'].iloc[0] if len(dup_sample) > 0 else None
if from_addr:
    all_matching = feature_table[feature_table['address_key'] == from_addr]
    dups_of_sample = all_matching[
        (all_matching['transaction_year'] == dup_sample['transaction_year'].iloc[0]) & 
        (all_matching['resale_price'] == dup_sample['resale_price'].iloc[0])
    ]
    print(f"   Address '{from_addr}' with same year/price: {len(dups_of_sample)} rows")
    if 'floor_area_sqm' in dups_of_sample.columns:
        print(f"   Unique floor areas: {dups_of_sample['floor_area_sqm'].unique()}")


DUPLICATE ROOT CAUSE ANALYSIS

1. KEY METRICS
   Total rows: 260,699
   Unique rows (all columns): 260,699
   Unique addresses: 9,710
   Avg rows per address: 26.8

2. HYPOTHESIS: Multiple flat models at same address
   Addresses with multiple rows: 9,645
   Addresses with exactly 1 row: 65

3. SAMPLE: Address '1 BEACH RD' (31 rows)
   Unique transactions years: 12
   Unique (year, price) combinations: 30
   All exact duplicates: True

4. ACTUAL DUPLICATE CHECK
   Showing 1 row that has duplicates:
   Columns: ['resale_price', 'transaction_year', 'level_mid', 'lease_remaining_years', 'floor_area_sqm', 'room_count', 'dist_to_mrt_m', 'orientation_score', 'dist_to_highway_m', 'dist_to_foodcourt_m']...
   Address '174 ANG MO KIO AVE 4' with same year/price: 1 rows
   Unique floor areas: [60.]
